# NB05 - Generator: FLUX.1-schnell  (`flux_schnell`)

**Run on its own Kaggle account, in parallel with the other generators.** Reads the frozen captions from NB02, generates one AI image per real image with **FLUX.1-schnell**, applies the *same* canonical preprocess used for the real images, and writes to `data/flux_schnell/`.

### Prerequisites
- NB00, NB01, NB02 finished (repo, library, config, real images, captions all exist).
- **Internet: ON. GPU: ON** (T4 x2 is fine).
- HF write token as the Kaggle secret `HF_TOKEN` on this account.

### Parallel-safe by design
- Writes only to `data/flux_schnell/` - never collides with the other five generators.
- Resumes from its own shards (counts what exists; `progress.json` is never trusted for counts).
- Seed = `hash(flux_schnell, source_real_id)`, so a crash/retry/double-run reproduces **identical** bytes.
- Commits one batch every 20 minutes; stops at exactly {target} accepted images.

To run the whole set: open this notebook on one account, and NB03-NB08 each on a different account. They can start on different days; NB09 only needs all six eventually finished.

In [1]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
# diffusers stack + hub/io. sentencepiece+protobuf are needed by the T5 text encoders (Flux/PixArt).
pip("diffusers>=0.30", "transformers>=4.40", "accelerate", "safetensors",
    "sentencepiece", "protobuf", "huggingface_hub>=0.23", "pyarrow", "pillow")
print("deps installed")


import importlib.util, sys, subprocess

def pip(*pkgs):
    # No -U flag -- upgrading Pillow/torchvision mid-session breaks Kaggle's preinstalled stack.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Only install packages that are missing or need a floor version.
# Pillow, torch, torchvision, transformers, accelerate, pyarrow, huggingface_hub are
# already on Kaggle's base image -- don't touch them.
to_check = {
    "diffusers":    "diffusers>=0.30",
    "safetensors":  "safetensors",
    "sentencepiece":"sentencepiece",
    "protobuf":     "protobuf",
}
missing = [pkg for mod, pkg in to_check.items() if importlib.util.find_spec(mod) is None]
if missing:
    pip(*missing)
    print("installed:", missing)
else:
    print("all packages already present -- no installs, no upgrades.")

import diffusers, torch
print(f"diffusers {diffusers.__version__} | torch {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.1/516.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
google-cloud-spanner 3.64

deps installed
installed: ['protobuf']
diffusers 0.38.0 | torch 2.10.0+cu128


In [2]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"          # same dataset repo as NB00
MODEL   = "flux_schnell"            # <-- this is the ONLY line that differs between NB03-NB08

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass
    return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()

# download + import the SAME shared library NB00 uploaded (identical canonical preprocess)
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
assert MODEL in cfg["generators"], f"{MODEL} not in config generators {list(cfg['generators'])}"
g = cfg["generators"][MODEL]
print("model:", MODEL, "| spec:", g)

ai_detect_common.py:   0%|          | 0.00/15.9k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.76k [00:00<?, ?B/s]

model: flux_schnell | spec: {'model_id': 'black-forest-labs/FLUX.1-schnell', 'pipeline': 'flux', 'native': 1024, 'steps': 4, 'guidance': 0.0}


In [3]:
import sys
import time
import torch
import torch.library

# =============================================================================
# PIL FIX
# =============================================================================

import PIL._typing

if not hasattr(PIL._typing, "_Ink"):
    from typing import Union
    PIL._typing._Ink = Union[int, float, str, bytes, tuple]

# =============================================================================
# TORCHVISION REGISTRATION FIX
# =============================================================================

if not getattr(torch.library, "_patched_for_torchvision", False):

    _orig_impl = torch.library.Library.impl

    def _safe_impl(self, *args, **kwargs):
        try:
            return _orig_impl(self, *args, **kwargs)
        except RuntimeError as e:
            if "already a kernel registered" in str(e):
                return
            raise

    torch.library.Library.impl = _safe_impl

    torch.library._patched_for_torchvision = True

for k in list(sys.modules):
    if k in ("PIL.ImageText", "PIL.ImageDraw"):
        del sys.modules[k]

# =============================================================================
# IMPORTS
# =============================================================================

from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    FluxPipeline,
    AutoPipelineForText2Image,
    PixArtSigmaPipeline,
    StableCascadePriorPipeline,
    StableCascadeDecoderPipeline,
)

# =============================================================================
# DTYPE
# =============================================================================

FLUX_DTYPE = torch.bfloat16
OTHER_DTYPE = torch.float16

# =============================================================================
# GENERATOR BUILDER
# =============================================================================

def build_generator(model_key, g):

    model_id = g["model_id"]
    native = g["native"]
    steps = g["steps"]
    guidance = g["guidance"]

    print(f"\nLoading: {model_key}")
    print(f"Model: {model_id}")

    start_time = time.time()

    # -------------------------------------------------------------------------
    # SD 1.5
    # -------------------------------------------------------------------------

    if model_key == "sd15":

        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            token=TOKEN,
            torch_dtype=OTHER_DTYPE,
            safety_checker=None,
            requires_safety_checker=False,
        )

        pipe.to("cuda")

        try:
            pipe.enable_vae_slicing()
        except:
            pass

        pipe.set_progress_bar_config(disable=True)

        def gen(prompt, seed):

            generator = torch.Generator(
                device="cpu"
            ).manual_seed(seed)

            return pipe(
                prompt,
                height=native,
                width=native,
                num_inference_steps=steps,
                guidance_scale=guidance,
                generator=generator,
            ).images[0]

        print(f"Loaded in {time.time()-start_time:.1f}s")
        return gen

    # -------------------------------------------------------------------------
    # SDXL
    # -------------------------------------------------------------------------

    elif model_key == "sdxl":

        pipe = StableDiffusionXLPipeline.from_pretrained(
            model_id,
            token=TOKEN,
            torch_dtype=OTHER_DTYPE,
            use_safetensors=True,
            add_watermarker=False,
        )

        pipe.enable_model_cpu_offload()
        pipe.set_progress_bar_config(disable=True)

        def gen(prompt, seed):

            generator = torch.Generator(
                device="cpu"
            ).manual_seed(seed)

            return pipe(
                prompt=prompt,
                height=native,
                width=native,
                num_inference_steps=steps,
                guidance_scale=guidance,
                generator=generator,
            ).images[0]

        print(f"Loaded in {time.time()-start_time:.1f}s")
        return gen

    # -------------------------------------------------------------------------
    # FLUX
    # -------------------------------------------------------------------------

    elif model_key == "flux_schnell":

        pipe = FluxPipeline.from_pretrained(
            model_id,
            token=TOKEN,
            torch_dtype=FLUX_DTYPE,
        )

        pipe.enable_sequential_cpu_offload()

        try:
            pipe.vae.enable_slicing()
            pipe.vae.enable_tiling()
        except:
            pass

        try:
            pipe.transformer.to(
                memory_format=torch.channels_last
            )
        except:
            pass

        pipe.set_progress_bar_config(disable=True)

        print(
            f"FLUX loaded in {time.time()-start_time:.1f}s"
        )

        def gen(prompt, seed):

            generator = torch.Generator(
                device="cpu"
            ).manual_seed(seed)

            image = pipe(
                prompt=prompt,
                height=native,
                width=native,
                num_inference_steps=steps,
                guidance_scale=guidance,
                max_sequence_length=256,
                generator=generator,
            ).images[0]

            return image

        return gen

    # -------------------------------------------------------------------------
    # Kandinsky
    # -------------------------------------------------------------------------

    elif model_key == "kandinsky22":

        pipe = AutoPipelineForText2Image.from_pretrained(
            model_id,
            token=TOKEN,
            torch_dtype=OTHER_DTYPE,
        )

        pipe.enable_model_cpu_offload()
        pipe.set_progress_bar_config(disable=True)

        def gen(prompt, seed):

            generator = torch.Generator(
                device="cpu"
            ).manual_seed(seed)

            return pipe(
                prompt=prompt,
                height=native,
                width=native,
                num_inference_steps=steps,
                guidance_scale=guidance,
                generator=generator,
            ).images[0]

        print(f"Loaded in {time.time()-start_time:.1f}s")
        return gen

    # -------------------------------------------------------------------------
    # PixArt
    # -------------------------------------------------------------------------

    elif model_key == "pixart_sigma":

        pipe = PixArtSigmaPipeline.from_pretrained(
            model_id,
            token=TOKEN,
            torch_dtype=OTHER_DTYPE,
            use_safetensors=True,
        )

        pipe.enable_model_cpu_offload()
        pipe.set_progress_bar_config(disable=True)

        def gen(prompt, seed):

            generator = torch.Generator(
                device="cpu"
            ).manual_seed(seed)

            return pipe(
                prompt=prompt,
                height=native,
                width=native,
                num_inference_steps=steps,
                guidance_scale=guidance,
                generator=generator,
            ).images[0]

        print(f"Loaded in {time.time()-start_time:.1f}s")
        return gen

    # -------------------------------------------------------------------------
    # Stable Cascade
    # -------------------------------------------------------------------------

    elif model_key == "sd_cascade":

        prior = StableCascadePriorPipeline.from_pretrained(
            g["prior_id"],
            token=TOKEN,
            torch_dtype=OTHER_DTYPE,
        )

        decoder = StableCascadeDecoderPipeline.from_pretrained(
            model_id,
            token=TOKEN,
            torch_dtype=OTHER_DTYPE,
        )

        prior.enable_model_cpu_offload()
        decoder.enable_model_cpu_offload()

        p_steps, d_steps = steps
        p_guid, d_guid = guidance

        def gen(prompt, seed):

            generator = torch.Generator(
                device="cpu"
            ).manual_seed(seed)

            prior_output = prior(
                prompt=prompt,
                negative_prompt=" ",
                height=native,
                width=native,
                guidance_scale=p_guid,
                num_inference_steps=p_steps,
                generator=generator,
            )

            return decoder(
                image_embeddings=prior_output.image_embeddings,
                prompt=prompt,
                negative_prompt=" ",
                guidance_scale=d_guid,
                num_inference_steps=d_steps,
                output_type="pil",
                generator=generator,
            ).images[0]

        print(f"Loaded in {time.time()-start_time:.1f}s")
        return gen

    else:
        raise ValueError(f"Unknown model: {model_key}")


generate = build_generator(MODEL, g)

print(f"\nPipeline ready -> {MODEL}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.



Loading: flux_schnell
Model: black-forest-labs/FLUX.1-schnell


model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

FLUX loaded in 142.3s

Pipeline ready -> flux_schnell


In [ ]:
import gc, torch
api = C.HfApi(token=TOKEN)
TARGET = cfg["target_per_generator"]
native = g["native"]
g_steps = int(g["steps"][0]) if isinstance(g["steps"], (list, tuple)) else int(g["steps"])
g_guid  = float(g["guidance"][0]) if isinstance(g["guidance"], (list, tuple)) else float(g["guidance"])

# 1. load the FROZEN captions table (written by NB02) -> {source_real_id: caption}
captions = {}
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id", "caption"])
    for sid, cap in zip(t.column("source_real_id").to_pylist(), t.column("caption").to_pylist()):
        captions[sid] = cap
print("captions available:", len(captions))
assert captions, "No captions found - run NB02 first."

# 2. resume: which source_real_ids this model already generated (authoritative = the shards)
done = C.reconcile_ids(REPO_ID, f"data/{MODEL}/", TOKEN)
todo = sorted(set(captions) - done)          # sorted -> deterministic order across sessions
print(f"already done: {len(done)} | remaining: {len(todo)} | target: {TARGET}")

# 3. generate
writer = C.ShardWriter(api, REPO_ID, f"data/{MODEL}/",
                       commit_interval=cfg["commit_interval_seconds"],
                       max_rows=cfg["batch_size"], token=TOKEN)
accepted, failed = 0, 0
try:
    for i, sid in enumerate(todo):
        if len(done) + accepted >= TARGET:
            print("target reached."); break
        prompt = captions[sid]
        seed = C.make_seed(MODEL, sid)        # deterministic: re-runs reproduce identical bytes
        try:
            img = generate(prompt, seed)               # native-resolution PIL
            png = C.canonical_preprocess(img)          # SAME transform as the real images -> 512 PNG
        except Exception as e:
            failed += 1
            if failed <= 20 or failed % 50 == 0:
                print(f"  gen failed for {sid} ({type(e).__name__}: {e}); will retry on a later run")
            continue
        num = str(sid).split("_")[-1]                  # real_000123 -> 000123
        row = C.empty_row()
        row.update(dict(image_id=f"{MODEL}_{num}", source_real_id=sid, label=1, generator=MODEL,
                        source_dataset=MODEL, prompt=prompt, image=png,
                        width=C.TARGET, height=C.TARGET, orig_width=native, orig_height=native,
                        gen_model_id=g["model_id"], gen_steps=g_steps, gen_guidance=g_guid,
                        gen_native_res=native, seed=int(seed), caption_model=cfg["caption_model"],
                        pipeline_version=C.PIPELINE_VERSION, sha256=C.sha256_bytes(png),
                        created_utc=C.now_utc()))
        writer.add(row); accepted += 1
        if accepted % 100 == 0:
            print(f"  {MODEL}: accepted {accepted} this run (total ~{len(done)+accepted}/{TARGET}); failed {failed}")
            gc.collect(); torch.cuda.empty_cache()
        writer.maybe_flush()
finally:
    writer.close()
print(f"done. accepted this run: {accepted}, failed: {failed}, total now ~{len(done)+accepted}/{TARGET}")

captions/captions-1dccecad-00000.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

captions/captions-1dccecad-00001.parquet:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

captions/captions-1dccecad-00002.parquet:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

captions/captions-1dccecad-00003.parquet:   0%|          | 0.00/23.0k [00:00<?, ?B/s]

captions/captions-1dccecad-00004.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

...

captions/captions-1dccecad-00096.parquet:   0%|          | 0.00/24.4k [00:00<?, ?B/s]

captions/captions-1dccecad-00097.parquet:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

captions/captions-1dccecad-00098.parquet:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

captions/captions-70411349-00000.parquet:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

captions available: 50000
already done: 0 | remaining: 50000 | target: 50000


## Self-check

In [ ]:
# quick self-check for THIS generator
n = C.count_rows(REPO_ID, f"data/{MODEL}/", TOKEN)
print(f"{MODEL}: {n} rows in data/{MODEL}/ (target {cfg['target_per_generator']})")
# decode one to confirm it is a canonical 512 PNG paired to a real id
import random
shards = C.list_shards(REPO_ID, f"data/{MODEL}/", TOKEN)
if shards:
    loc = C.hf_hub_download(REPO_ID, shards[-1], repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(loc, columns=["image", "image_id", "source_real_id", "label"])
    j = random.randrange(t.num_rows)
    ok, why = C.png_is_canonical(t.column("image")[j].as_py())
    print("sample:", t.column("image_id")[j].as_py(), "<-", t.column("source_real_id")[j].as_py(),
          "| label", t.column("label")[j].as_py(), "| canonical", ok, why)
print("RESULT:", "looks correct" if n > 0 else "no rows yet")